In [3]:

import pandas as pd

# Load the dataset
df = pd.read_csv(
    "D:/Customer-Churn-Analysis/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

# Display the first five rows
print(df.head())

# Check the number of rows and columns
print("Dataset shape:", df.shape)

# Inspect column names and data types
df.info()

# Count missing values in each column
print(df.isnull().sum())

# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())


   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [4]:

# Check the current data type
print(df["TotalCharges"].dtype)

# Display some raw values
print(df["TotalCharges"].head(15).tolist())

# Count blank or whitespace-only strings
blank_total_charges = (
    df["TotalCharges"]
    .astype("string")
    .str.strip()
    .eq("")
    .sum()
)

print("Blank TotalCharges values:", blank_total_charges)


object
['29.85', '1889.5', '108.15', '1840.75', '151.65', '820.5', '1949.4', '301.9', '3046.05', '3487.95', '587.45', '326.8', '5681.1', '5036.3', '2686.05']
Blank TotalCharges values: 11


In [5]:

# Remove surrounding whitespace and convert invalid values to NaN
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].astype("string").str.strip(),
    errors="coerce"
)

# Verify the conversion
print("New data type:", df["TotalCharges"].dtype)

# Count missing values after conversion
print(
    "Missing TotalCharges values:",
    df["TotalCharges"].isnull().sum()
)


New data type: Float64
Missing TotalCharges values: 11


In [6]:

# Inspect customers with missing TotalCharges
missing_charges = df[df["TotalCharges"].isnull()]

display(
    missing_charges[
        ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
    ]
)

print("Rows with missing charges:", len(missing_charges))

print(
    "Missing-charge rows with zero tenure:",
    (missing_charges["tenure"] == 0).sum()
)


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,<NA>,No
753,3115-CZMZD,0,20.25,<NA>,No
936,5709-LVOEQ,0,80.85,<NA>,No
1082,4367-NUYAO,0,25.75,<NA>,No
1340,1371-DWPAZ,0,56.05,<NA>,No
3331,7644-OMVMY,0,19.85,<NA>,No
3826,3213-VVOLG,0,25.35,<NA>,No
4380,2520-SGTTA,0,20.00,<NA>,No
5218,2923-ARZLG,0,19.70,<NA>,No
6670,4075-WKNIU,0,73.35,<NA>,No


Rows with missing charges: 11
Missing-charge rows with zero tenure: 11


In [7]:

# Remove rows where TotalCharges is missing
df = df.dropna(subset=["TotalCharges"]).copy()

print("Shape after handling missing charges:", df.shape)
print("Remaining missing TotalCharges:", df["TotalCharges"].isnull().sum())


Shape after handling missing charges: (7032, 21)
Remaining missing TotalCharges: 0


In [8]:

# Clean surrounding whitespace in text columns
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for column in text_columns:
    df[column] = df[column].str.strip()

# Check whether Churn has the expected categories
print(df["Churn"].value_counts(dropna=False))



Churn
No     5163
Yes    1869
Name: count, dtype: int64


In [9]:

print("Final dataset shape:", df.shape)

print("\nTotal missing values:", df.isnull().sum().sum())

print("Duplicate rows:", df.duplicated().sum())

print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

print("TotalCharges data type:", df["TotalCharges"].dtype)

print("\nChurn distribution:")
print(df["Churn"].value_counts(dropna=False))

print("\nTenure summary:")
print(df["tenure"].describe())

print("\nMonthly charges summary:")
print(df["MonthlyCharges"].describe())


Final dataset shape: (7032, 21)

Total missing values: 0
Duplicate rows: 0
Duplicate customer IDs: 0
TotalCharges data type: Float64

Churn distribution:
Churn
No     5163
Yes    1869
Name: count, dtype: int64

Tenure summary:
count    7032.000000
mean       32.421786
std        24.545260
min         1.000000
25%         9.000000
50%        29.000000
75%        55.000000
max        72.000000
Name: tenure, dtype: float64

Monthly charges summary:
count    7032.000000
mean       64.798208
std        30.085974
min        18.250000
25%        35.587500
50%        70.350000
75%        89.862500
max       118.750000
Name: MonthlyCharges, dtype: float64


In [10]:

# Define the output path
output_path = (
    r"D:\Customer-Churn-Analysis\data\processed\customer_churn_cleaned.csv"
)

# Save the cleaned DataFrame
df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully.")
print("Location:", output_path)


Cleaned dataset saved successfully.
Location: D:\Customer-Churn-Analysis\data\processed\customer_churn_cleaned.csv
